**Feature Selection Method:** CORR   
**Dataset:** QUT-DV25 (SysCall)   
**Developed By:** eDySec Research Team   
**Plartform:** Ubuntu

All experiments in this notebook were conducted using **Python 3.10** with the following libraries:

`pandas==1.5.3`,  
`scikit-learn==1.2.2`,  
`openpyxl`,  
`numpy==1.23.5`,  
`scipy==1.9.3`,  
`tensorflow==2.11.0`,  
`matplotlib==3.7.1`,  
`seaborn==0.12.2`,  
`joblib==1.3.2`,  
`shap==0.41.0`,  
`lime`,  
`flaml[automl]==2.5.0`,  
`notebook==6.5.6`,  
`pywinpty==2.0.10`  (Only for windows)  `threadpoolctl==3.1.0` (for Ubuntu)   
`terminado==0.17.1`,  
`transformers==4.49.0`.

#### Full Environment Setup: https://github.com/tanzirmehedi/eDySec

These versions were used to ensure **consistent and reproducible experimental results**.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from scipy.sparse import hstack
from scipy.sparse import csr_matrix
from sklearn.metrics import (
    accuracy_score, confusion_matrix, roc_auc_score, roc_curve,
    precision_recall_fscore_support, cohen_kappa_score
)
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
import time
import joblib
import gc
from sklearn.feature_selection import f_classif

In [ ]:

# This will allow TensorFlow to allocate as much GPU memory as needed, including full 16GB if needed.
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # Disable memory growth (optional - default is False)
        tf.config.experimental.set_memory_growth(gpus[0], False)
        print("Memory growth disabled (default behavior).")
    except RuntimeError as e:
        print("Error:", e)

Memory growth disabled (default behavior).


In [ ]:
file_path = 'QUT-DV25_'+DATASET_NAME+'_Traces.csv'
data = pd.read_csv(file_path)

In [ ]:
gc.collect()
tf.keras.backend.clear_session()

In [ ]:
data.head()

,Package_Name,Total_System_Calls,Unique_System_Calls,Unique_System_Calls_List,File_Operations,Unique_File_Operations,Unique_File_Operations_List,Memory_Operations,Unique_Memory_Operations,Unique_Memory_Operations_List,...,Filesystem_Operations,Unique_Filesystem_Operations,Unique_Filesystem_Operations_List,Security_Operations,Unique_Security_Operations,Unique_Security_Operations_List,Miscellaneous_Operations,Unique_Miscellaneous_Operations,Unique_Miscellaneous_Operations_List,Level
0,10Cent10-999.0.4.tar.gz,4859,33,"munmap, newfstatat, openat, fstat, ioctl, lsee...",3945,12,"newfstatat, mkdir, fstat, write, unlink, read,...",433,4,"munmap, mmap, mprotect, brk",...,0,0,NaN,0,0,NaN,7,2,"getrandom, uname",1
1,10Cent11-999.0.4.tar.gz,1423,34,"read, getpid, ioctl, write, wait4, close, newf...",1014,8,"newfstatat, write, fstat, read, rmdir, lseek, ...",25,4,"munmap, brk, mprotect, mmap",...,0,0,NaN,0,0,NaN,2,1,uname,1
2,11Cent-999.0.0.tar.gz,9966,42,"newfstatat, mkdir, openat, fstat, write, close...",8556,12,"newfstatat, mkdir, fstat, write, unlink, read,...",204,5,"mremap, munmap, brk, mmap, mprotect",...,0,0,NaN,0,0,NaN,8,2,"getrandom, uname",1
3,11Cent-999.0.1.tar.gz,1049,33,"restart_syscall, getsockopt, newfstatat, opena...",782,8,"newfstatat, openat, fstat, write, read, rmdir,...",11,3,"munmap, brk, mmap",...,0,0,NaN,0,0,NaN,2,1,uname,1
4,11Cent-999.0.2.tar.gz,791,33,"restart_syscall, read, brk, poll, uname, newfs...",625,9,"newfstatat, fstat, write, unlink, read, rmdir,...",5,3,"munmap, brk, mmap",...,0,0,NaN,0,0,NaN,2,1,uname,1


In [ ]:

# X = your feature DataFrame, y = target column
X = data.drop(columns=['Level'])
y = data['Level']

# 1. Compute correlation with target
correlation_with_target = X.apply(lambda col: col.corr(y) if np.issubdtype(col.dtype, np.number) else 0)

# 2. Drop features highly correlated with each other (|r| >= 0.5)
corr_matrix = X.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Features to drop due to high correlation
to_drop = [column for column in upper.columns if any(upper[column] >= 0.5)]

# 3. Keep features with |r| < 0.5
selected_features = [f for f in X.columns if f not in to_drop]

# 4. If you have feature importance scores (IMS) from ML models, apply threshold
# Example: keep features with IMS > 0.05 in at least one model
# Assuming ims_dict = {'feature_name': max_importance_across_models, ...}
# selected_features = [f for f in selected_features if ims_dict.get(f, 0) > 0.05]

# 5. Final selected features DataFrame
X_selected_df = X[selected_features]

print("Selected features:", list(X_selected_df.columns))
print("Number of features kept:", X_selected_df.shape[1])
print("Number of features dropped:", X.shape[1] - X_selected_df.shape[1])


Selected features: ['Total_System_Calls', 'Unique_System_Calls', 'Unique_System_Calls_List', 'Unique_File_Operations_List', 'Memory_Operations', 'Unique_Memory_Operations_List', 'Unique_Network_Operations_List', 'Process_Management_Operations', 'Unique_Process_Management_Operations_List', 'Unique_IO_Operations_List', 'Time_Operations', 'Unique_Time_Operations', 'Unique_Time_Operations_List', 'Unique_IPC_Operations_List', 'Filesystem_Operations', 'Unique_Filesystem_Operations_List', 'Security_Operations', 'Unique_Security_Operations', 'Unique_Security_Operations_List', 'Unique_Miscellaneous_Operations_List', 'Combined_Categorical']
Number of features kept: 21
Number of features dropped: 13


In [ ]:
selected_features = ['Total_System_Calls', 'Unique_System_Calls', 'Unique_System_Calls_List', 'Unique_File_Operations_List', 'Memory_Operations', 'Unique_Memory_Operations_List', 'Unique_Network_Operations_List', 'Process_Management_Operations', 'Unique_Process_Management_Operations_List', 'Unique_IO_Operations_List', 'Time_Operations', 'Unique_Time_Operations', 'Unique_Time_Operations_List', 'Unique_IPC_Operations_List', 'Filesystem_Operations', 'Unique_Filesystem_Operations_List', 'Security_Operations', 'Unique_Security_Operations', 'Unique_Security_Operations_List', 'Unique_Miscellaneous_Operations_List']